# Table of Contents
- [Description](#description)
- [Read in data file](#read-in-data-file)
- [Map ISCO 08 Codes to O*NET-SOC 2019](#map-isco-08-codes-in-dataframe-to-onet-soc-2019)
- [Inspecting ISCO 08 and O*NET-SOC 2019 missingness](#inspecting-isco-08-and-onet-soc-2019-missingness)
- [General info about missing values](#general-info-about-missing-values)

# Description
The purpose of this notebook is to construct our ISSP-based dataset, primarily by: 
1. Taking an initial look at the variables that the dataset offers,
2. Renaming them to be more readable,
3. Mapping ISCO 08 codes to O*NET-SOC 2019

And then saving the resulting DataFrame off to a CSV under the `/data/constructed_datasets/` folder.

## Read in data file

In [1]:
import pandas as pd
import pyreadstat

data_dir = "../../data/"
spss_dir = data_dir + "spss_data/"
path = spss_dir + "ZA8797_v1-0-0.sav"

_, meta = pyreadstat.read_sav(path, metadataonly=True)

In [2]:
meta.original_variable_types

{'studyno': 'F4.0',
 'version': 'A18',
 'doi': 'A31',
 'caseid': 'F16.0',
 'cumu_id': 'F16.0',
 'year': 'F4.0',
 'country': 'F3.0',
 'c_sample': 'F5.0',
 'c_sample_year': 'F9.0',
 'v1': 'F1.0',
 'v2': 'F1.0',
 'v3': 'F1.0',
 'v4': 'F1.0',
 'v5': 'F1.0',
 'v6': 'F1.0',
 'v7': 'F1.0',
 'v8': 'F1.0',
 'v9': 'F1.0',
 'v10': 'F1.0',
 'v11': 'F1.0',
 'v12': 'F1.0',
 'v13': 'F1.0',
 'v14': 'F1.0',
 'v15': 'F1.0',
 'v16': 'F1.0',
 'v17': 'F1.0',
 'v18': 'F1.0',
 'v19': 'F1.0',
 'v20': 'F1.0',
 'v21': 'F1.0',
 'v22': 'F1.0',
 'v23': 'F1.0',
 'v24': 'F1.0',
 'v25': 'F1.0',
 'v26': 'F1.0',
 'v27': 'F1.0',
 'v28': 'F1.0',
 'v29': 'F1.0',
 'v30': 'F1.0',
 'v31': 'F1.0',
 'v32': 'F1.0',
 'v33': 'F1.0',
 'v34': 'F1.0',
 'v35': 'F1.0',
 'v36': 'F1.0',
 'v37': 'F1.0',
 'v38': 'F1.0',
 'v39': 'F1.0',
 'v40': 'F1.0',
 'v41': 'F1.0',
 'v42': 'F1.0',
 'v43': 'F1.0',
 'v44': 'F1.0',
 'v45': 'F1.0',
 'v46': 'F1.0',
 'v47': 'F1.0',
 'v48': 'F1.0',
 'v49': 'F1.0',
 'v50': 'F1.0',
 'v51': 'F1.0',
 'v52': 'F1.0'

In [3]:
meta.column_names_to_labels

{'studyno': 'GESIS Study Number',
 'version': 'GESIS Archive Version',
 'doi': 'Digital Object Identifier',
 'caseid': 'ID Number of Respondent',
 'cumu_id': 'Unique Cumulation Respondent ID Number',
 'year': 'ISSP ModuleYear_StudyNumber',
 'country': 'Country',
 'c_sample': 'Country Sample',
 'c_sample_year': 'Country Sample Year',
 'v1': 'abc: Spending time: time in a paid job',
 'v2': 'abc: Spending time: in doing household work',
 'v3': 'abc: Spending time: with the family',
 'v4': 'abc: Spending time: with friends',
 'v5': 'abc: Spending time: in leisure activities',
 'v6': 'abcd: Job is a way of earning money',
 'v7': 'abcd: Enjoy a paid job even if I did not need money',
 'v8': 'ab: Work most important activity',
 'v9': 'ab: Respondent domestic duties',
 'v10': 'ad: Workers need trade unions',
 'v11': 'abcd: Personally important: job security',
 'v12': 'abcd: Personally important: high income',
 'v13': 'abcd: Personally important: opportunities for advancement',
 'v14': 'abcd: P

In [4]:
rename_map = {
    # --- identifiers / metadata ---
    "studyno":       "study_no",
    "version":       "archive_version",
    "doi":           "doi",
    "caseid":        "case_id",
    "cumu_id":       "cumu_id",
    "year":          "module_year",
    "country":       "country",
    "c_sample":      "country_sample",
    "c_sample_year": "country_sample_year",

    # --- time use (a,b,c only -- NOT asked in 2015) ---
    "v1": "time_paid_job",
    "v2": "time_housework",
    "v3": "time_family",
    "v4": "time_friends",
    "v5": "time_leisure",

    # --- work centrality (abcd unless noted) ---
    "v6":  "job_just_for_money",
    "v7":  "would_work_without_need",
    "v8":  "work_most_important_activity",   # a,b only
    "v9":  "domestic_duties",                # a,b only
    "v10": "unions_needed",                  # a,d only

    # --- IMPORTANCE battery: personally important in a job (all abcd) ---
    "v11": "imp_job_security",
    "v12": "imp_income",
    "v13": "imp_advancement",
    "v14": "imp_interesting",
    "v15": "imp_independence",
    "v16": "imp_help_others",
    "v17": "imp_useful_society",
    "v18": "imp_decide_hours",

    "v19": "choice_job_kinds",       # a,b,c only
    "v20": "choice_firm_kinds",      # a,b,c only
    "v21": "choice_worktypes",       # a,b,c only
    "v22": "easy_find_acceptable_job",  # c,d only
    "v23": "how_hard_works",         # a,b only
    "v24": "pref_hours_vs_earnings",

    # --- ACTUAL JOB battery: does this apply to R's current job (all abcd) ---
    "v25": "actual_job_secure",
    "v26": "actual_income_high",
    "v27": "actual_advancement_high",
    "v28": "actual_job_interesting",
    "v29": "actual_work_independently",
    "v30": "actual_help_others",
    "v31": "actual_useful_society",

    # --- frequency / strain items ---
    "v32": "freq_exhausted_after_work",     # a,b,c only
    "v33": "freq_hard_physical_work",
    "v34": "freq_work_stressful",
    "v35": "freq_dangerous_conditions",     # a,b,c only
    "v36": "freq_job_interferes_family",    # c,d only
    "v37": "freq_family_interferes_job",    # c,d only

    "v38": "relations_mgmt_employees",
    "v39": "relations_workmates",

    # --- SATISFACTION (all abcd) ---
    "v40": "job_satisfaction",

    "v41": "second_job",             # a,c only
    "v42": "pref_work_situation",    # b,c,d only
    "v43": "currently_paid_work",    # b,c,d only
    "v44": "working_hours_conditions",   # b,c,d only
    "v45": "daily_work_organization",    # c,d only
    "v46": "difficulty_time_off",        # c,d only
    "v47": "use_past_experience_skills", # b,c,d only
    "v48": "training_past_12mo",         # c,d only

    "v49": "willing_work_harder",    # b,c,d only
    "v50": "proud_of_firm",          # b,c,d only
    "v51": "would_change_type_of_work",   # b,d only
    "v52": "would_turn_down_other_job",   # b,c,d only
    "v53": "proud_of_work_type",     # b,d only
    "v54": "likely_seek_new_job_12mo",    # b,c,d only
    "v55": "worry_losing_job",       # b,c,d only

    # --- not-currently-working sub-battery (b,c,d only) ---
    "v56": "ever_had_paid_job_1yr",
    "v57": "year_last_job_ended",
    "v58": "reason_job_ended",
    "v59": "want_paid_job",
    "v60": "likely_find_job",
    "v61": "currently_looking_for_job",
    "v62": "registered_public_agency",
    "v63": "registered_private_agency",
    "v64": "answered_job_ad",
    "v65": "advertised_self",
    "v66": "applied_directly_employers",
    "v67": "asked_relatives_friends",
    "v68": "main_economic_support",
    "v69": "training_not_working_12mo",   # c,d only
    "v70": "avoid_unemp_new_skills",       # c,d only
    "v71": "avoid_unemp_lower_pay",        # c,d only
    "v72": "avoid_unemp_temp_employment",  # c,d only
    "v73": "avoid_unemp_travel_longer",    # c,d only

    # --- demographics / background ---
    "SEX":      "sex",
    "AGE":      "age",
    "MARITAL":  "marital_status",
    "COHAB":    "steady_partner",
    "EDUCYRS":  "educ_years",
    "DEGREE":   "educ_level",
    "WORK":     "work_status_ever",
    "EMPREL":   "employment_relationship",
    "WRKHRS":   "hours_worked_weekly",
    "WRKSUP":   "supervises_others",
    "NSUP":     "n_supervised",
    "TYPORG2":  "org_type_public_private",
    "ISCO08":   "isco08",
    "MAINSTAT": "employment_status_current",

    # --- spouse/partner (only needed for household-level analysis) ---
    "SPWORK":   "partner_work_status",
    "SPEMPREL": "partner_employment_relationship",
    "SPISCO08": "partner_isco08",
    "SPMAINST": "partner_employment_status",

    "UNION":    "union_member",
    "HOMPOP":   "household_size",
    "CHILDHH":  "child_in_household",
    "PARTY_LR": "party_left_right",
    "VOTE_LE":  "voted_last_election",
    "ATTEND":   "religious_attendance",
    "RELIGGRP": "religion_group",
    "TOPBOT":   "social_status_scale",
    "CLASS":    "subjective_class",
    "URBRURAL": "urban_rural",
    "INCOME":   "household_income_group",
    "WEIGHT":   "weight",
    "MODE":     "data_collection_mode",
}

In [5]:
df, meta2 = pyreadstat.read_sav(path, user_missing=False)

In [6]:
df = df.rename(columns=rename_map)
df.head()

,study_no,archive_version,doi,case_id,cumu_id,module_year,country,country_sample,country_sample_year,time_paid_job,...,party_left_right,voted_last_election,religious_attendance,religion_group,social_status_scale,subjective_class,urban_rural,household_income_group,weight,data_collection_mode
0,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000001.0,2005000360000001,2005.0,36.0,36.0,362005.0,5.0,...,2.0,1.0,5.0,2.0,2.0,NaN,5.0,NaN,1.0,34.0
1,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000002.0,2005000360000002,2005.0,36.0,36.0,362005.0,1.0,...,2.0,1.0,6.0,2.0,3.0,NaN,2.0,NaN,1.0,34.0
2,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000003.0,2005000360000003,2005.0,36.0,36.0,362005.0,4.0,...,2.0,1.0,2.0,2.0,6.0,NaN,1.0,2.0,1.0,34.0
3,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000004.0,2005000360000004,2005.0,36.0,36.0,362005.0,4.0,...,2.0,1.0,5.0,2.0,7.0,NaN,1.0,3.0,1.0,34.0
4,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000005.0,2005000360000005,2005.0,36.0,36.0,362005.0,4.0,...,2.0,1.0,5.0,2.0,8.0,NaN,2.0,3.0,1.0,34.0


## Map ISCO 08 codes in DataFrame to O*NET-SOC 2019

In [7]:
df_isco_onet = pd.read_csv(data_dir + "crosswalks/isco2008_to_onet2019.csv")
df_isco_onet.drop(columns=["ISCO-08 Title EN", 
                           "O*NET-SOC 2019 Title"], 
                  inplace=True)

df_isco_onet.head()

,ISCO-08 Code,O*NET-SOC 2019 Code
0,1111,11-1031.00
1,1211,11-3031.00
2,1213,13-1082.00
3,1311,11-9013.00
4,1312,11-9013.00


In [9]:
isco_to_onet = {isco:onet for isco,onet 
                in zip(df_isco_onet["ISCO-08 Code"],
                       df_isco_onet["O*NET-SOC 2019 Code"])}

df["ONET_SOC_CODE"] = df["isco08"].map(isco_to_onet)

In [10]:
df.head()

,study_no,archive_version,doi,case_id,cumu_id,module_year,country,country_sample,country_sample_year,time_paid_job,...,voted_last_election,religious_attendance,religion_group,social_status_scale,subjective_class,urban_rural,household_income_group,weight,data_collection_mode,ONET_SOC_CODE
0,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000001.0,2005000360000001,2005.0,36.0,36.0,362005.0,5.0,...,1.0,5.0,2.0,2.0,NaN,5.0,NaN,1.0,34.0,NaN
1,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000002.0,2005000360000002,2005.0,36.0,36.0,362005.0,1.0,...,1.0,6.0,2.0,3.0,NaN,2.0,NaN,1.0,34.0,45-1011.00
2,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000003.0,2005000360000003,2005.0,36.0,36.0,362005.0,4.0,...,1.0,2.0,2.0,6.0,NaN,1.0,2.0,1.0,34.0,NaN
3,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000004.0,2005000360000004,2005.0,36.0,36.0,362005.0,4.0,...,1.0,5.0,2.0,7.0,NaN,1.0,3.0,1.0,34.0,43-3031.00
4,8797.0,1.0.0 (2024-10-07),https://doi.org/10.4232/1.14391,1000005.0,2005000360000005,2005.0,36.0,36.0,362005.0,4.0,...,1.0,5.0,2.0,8.0,NaN,2.0,3.0,1.0,34.0,11-3071.00


## Inspecting ISCO 08 and O*NET-SOC 2019 missingness
ISCO 08 codes that do not have an O\*NET-SOC 2019 code corresponding to it is because they are more general job codes. As a result, it is not present in the crosswalk from ISCO 08 to SOC 2010 that was used to map between ISCO 08 and O\*NET-SOC 2019.

It will be left up to the team member who is cleaning the data to decide how best to handle this. 

In [11]:
has_isco = df["isco08"].notna()
no_onet = df["ONET_SOC_CODE"].isna()

isco_with_no_mapping = df[has_isco & no_onet]["isco08"].unique()

print(f"[INFO] {len(isco_with_no_mapping)} ISCO-2008 codes do not have an O*NET-SOC 2019 mapping")
isco_with_no_mapping

[INFO] 150 ISCO-2008 codes do not have an O*NET-SOC 2019 mapping


array([5320., 2500., 3340., 4220., 1410., 2640., 8100., 2210., 4100.,
       5130., 9000., 2300., 3110., 3510., 3150.,  300., 2400., 2220.,
       2410., 2260., 2130., 9210., 3420., 6100., 8330., 1300., 5150.,
       1400., 8210., 2110., 7210., 2630., 7230., 2140., 2160., 5140.,
       7320., 4410., 3000., 3430., 3350., 7200., 9310., 8310., 7400.,
       1200., 9330., 8340., 1000., 3100., 8150., 9610., 9320., 7000.,
       3250., 7510., 2200., 3130., 5240., 9300., 5100., 4310., 3120.,
       8180., 9110., 9120., 1110., 6220., 7220., 5200., 1220., 2650.,
       5160., 6120., 7410., 7530., 3410., 5220., 2150., 1340., 8320.,
       8120., 7130., 3310., 7120., 3210., 4210., 5310., 2610., 2350.,
       3220., 4320., 2340., 7420., 7540., 1310., 4130., 7110., 1320.,
       9410., 7520., 6110., 3520., 3330., 9620., 1210., 2520., 3320.,
       8140., 5110., 5410., 2430., 2510., 5210., 8130., 2420., 8170.,
       6300., 8000., 3200., 9200.,  200., 8110., 3140., 3300., 4200.,
       5300., 2100.,

In [12]:
num_rows_affected = df[has_isco & no_onet].shape[0]
total_rows = df.shape[0]

print("[INFO] Number of rows that have an ISCO 08 code but no corresponding O*NET:",
      num_rows_affected)
print("[INFO] Percentage of total rows that are affected:",
      f"{num_rows_affected} / {total_rows}", 
      f"({(num_rows_affected / total_rows) * 100 :.2f}%)")

[INFO] Number of rows that have an ISCO 08 code but no corresponding O*NET: 21637
[INFO] Percentage of total rows that are affected: 21637 / 122003 (17.73%)


In [13]:
mask = has_isco & no_onet
df["isco08_no_onet_match"] = mask

print("Missingness by country:")
print(df.groupby("country")["isco08_no_onet_match"].mean().sort_values(ascending=False).head(15))

print()
print("Missingness by wave:")
print(df.groupby("module_year")["isco08_no_onet_match"].mean().sort_values(ascending=False))

print()
print("Missingness by ISCO major group (first digit, always known):")

ARMED_FORCES_STRIPPED = {"100", "200", "300", "110", "210", "310"}

def get_major_group(code):
    if pd.isna(code):
        return None
    s = str(int(float(code)))
    if s in ARMED_FORCES_STRIPPED:
        return "0"
    return s[0]

major_group = df["isco08"].apply(get_major_group)
print(df.groupby(major_group)["isco08_no_onet_match"].mean().sort_values(ascending=False))

Missingness by country:
country
124.0    0.547858
250.0    0.436576
40.0     0.372248
246.0    0.341444
620.0    0.300518
608.0    0.285000
705.0    0.266997
100.0    0.230300
376.0    0.211455
276.0    0.206588
710.0    0.204670
756.0    0.193749
348.0    0.193577
36.0     0.192560
724.0    0.174435
Name: isco08_no_onet_match, dtype: float64

Missingness by wave:
module_year
2005.0    0.253878
1997.0    0.214625
1989.0    0.096477
2015.0    0.090230
Name: isco08_no_onet_match, dtype: float64

Missingness by ISCO major group (first digit, always known):
isco08
0    0.652062
9    0.352443
1    0.314779
4    0.271053
2    0.262020
5    0.257494
3    0.236902
8    0.211834
7    0.180463
6    0.152709
Name: isco08_no_onet_match, dtype: float64


In [14]:
df.drop(columns="isco08_no_onet_match", inplace=True)

## General info about missing values

In [15]:
with pd.option_context('display.max_rows', None):
    print(df.isna().sum())

study_no                                0
archive_version                         0
doi                                     0
case_id                                 0
cumu_id                                 0
module_year                             0
country                                 0
country_sample                          0
country_sample_year                     0
time_paid_job                       52320
time_housework                      42184
time_family                         40808
time_friends                        40028
time_leisure                        40090
job_just_for_money                   4417
would_work_without_need              7003
work_most_important_activity        76839
domestic_duties                     76014
unions_needed                       75327
imp_job_security                     2862
imp_income                           2990
imp_advancement                      3930
imp_interesting                      2906
imp_independence                  

In [16]:
# Write to CSV
output_path = data_dir + "constructed_datasets/issp.csv"
df.to_csv(output_path, index=False)